In [ ]:
!wget -O blaze_face_short_range.tflite \
https://storage.googleapis.com/mediapipe-models/face_detector/blaze_face_short_range/float16/1/blaze_face_short_range.tflite


from google.colab import drive
drive.mount('/content/drive')

!pip uninstall -y torch torchvision torchaudio
!pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu121

!pip install mediapipe









--2026-01-16 01:22:21--  https://storage.googleapis.com/mediapipe-models/face_detector/blaze_face_short_range/float16/1/blaze_face_short_range.tflite
Resolving storage.googleapis.com (storage.googleapis.com)... 142.251.10.207, 142.251.12.207, 64.233.170.207, ...
Connecting to storage.googleapis.com (storage.googleapis.com)|142.251.10.207|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 229746 (224K) [application/octet-stream]
Saving to: ‘blaze_face_short_range.tflite’

blaze_face_short_ra 100%[===================>] 224.36K   389KB/s    in 0.6s    

2026-01-16 01:22:22 (389 KB/s) - ‘blaze_face_short_range.tflite’ saved [229746/229746]

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Found existing installation: torch 2.5.1+cu121
Uninstalling torch-2.5.1+cu121:
  Successfully uninstalled torch-2.5.1+cu121
Looking in indexes: https://download.pytorch.org/whl/cu121
  Using cached https

The following cells create the `utils` directory and two Python files: `preprocess.py` and `math.py`. These files contain placeholder functions that were imported in your main script. You will need to implement the detailed logic for these functions.

In [4]:
import os
import cv2
import torch
import torchvision
import numpy as np
from tqdm import tqdm
import mediapipe as mp

BaseOptions = mp.tasks.BaseOptions
FaceDetectorOptions = mp.tasks.vision.FaceDetectorOptions
VisionRunningMode = mp.tasks.vision.RunningMode

detector = mp.tasks.vision.FaceDetector.create_from_options(
    FaceDetectorOptions(
        base_options=BaseOptions(model_asset_path='/content/blaze_face_short_range.tflite'),
        running_mode=VisionRunningMode.IMAGE,
        min_detection_confidence=0.5
    )
)

DATASET_ROOT = "/content/drive/MyDrive/state-farm-distracted-driver-detection/imgs/train"
SAVE_ROOT = "/content/preprocessed_dataset"

os.makedirs(SAVE_ROOT, exist_ok=True)

classes = sorted([c for c in os.listdir(DATASET_ROOT) if os.path.isdir(os.path.join(DATASET_ROOT, c))])

for c in classes:
    class_dir = os.path.join(DATASET_ROOT, c)
    save_class_dir = os.path.join(SAVE_ROOT, c)
    os.makedirs(save_class_dir, exist_ok=True)

    for fname in tqdm(os.listdir(class_dir)):
        if not fname.lower().endswith(('.jpg', '.jpeg', '.png')):
            continue

        path = os.path.join(class_dir, fname)
        frame = cv2.imread(path)
        if frame is None:
            continue
        H, W, _ = frame.shape

        rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        mp_image = mp.Image(image_format=mp.ImageFormat.SRGB, data=rgb)
        result = detector.detect(mp_image)

        if result and result.detections and len(result.detections) > 0:
            face = result.detections[0].bounding_box
            x1 = max(0, int(face.origin_x - 0.25 * face.width))
            y1 = max(0, int(face.origin_y - 0.25 * face.height))
            x2 = min(W, int(face.origin_x + face.width + 0.25 * face.width))
            y2 = min(H, int(face.origin_y + face.height + 0.25 * face.height))
            face_roi = frame[y1:y2, x1:x2]
        else:
            face_roi = frame.copy()

        hand_roi = frame[H//2:H, W//2:W]
        full_roi = frame.copy()

        def preprocess(img):
            img = cv2.resize(img, (224, 224))
            img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
            img = img.astype(np.float32) / 255.0
            img = torch.from_numpy(img).permute(2, 0, 1).contiguous()
            mean = torch.tensor([0.485, 0.456, 0.406], dtype=img.dtype).view(3,1,1)
            std  = torch.tensor([0.229, 0.224, 0.225], dtype=img.dtype).view(3,1,1)
            return (img - mean) / std

        # Save preprocessed tensors
        save_path = os.path.join(save_class_dir, fname.split('.')[0] + ".pt")
        torch.save({
            "full": preprocess(full_roi),
            "face": preprocess(face_roi),
            "hand": preprocess(hand_roi)
        }, save_path)


100%|██████████| 2129/2129 [01:35<00:00, 22.37it/s]


In [5]:
import torch
from torch.utils.data import Dataset
import os

class DriverDataset(Dataset):
    def __init__(self, root_dir):
        self.samples = []
        classes = sorted([c for c in os.listdir(root_dir) if os.path.isdir(os.path.join(root_dir, c))])
        self.class_map = {c: idx for idx, c in enumerate(classes)}
        for c in classes:
            class_dir = os.path.join(root_dir, c)
            for fname in os.listdir(class_dir):
                if fname.endswith(".pt"):
                    self.samples.append((os.path.join(class_dir, fname), self.class_map[c]))

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        path, label = self.samples[idx]
        data = torch.load(path)
        return data["full"], data["face"], data["hand"], label


In [6]:
import torch.nn as nn

class DriverActionClassifier(nn.Module):
    def __init__(self, backbone, num_classes=10):
        super().__init__()
        self.backbone = backbone
        self.classifier = nn.Linear(3 * 576, num_classes)

    def forward(self, image, face, hand):
        im = self.backbone(image).flatten(1)
        f = self.backbone(face).flatten(1)
        ha = self.backbone(hand).flatten(1)
        combined = torch.cat([im, f, ha], dim=1)
        return self.classifier(combined)


In [7]:
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, random_split
import torchvision.models as models

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

DATASET_ROOT = "/content/preprocessed_dataset"
dataset = DriverDataset(DATASET_ROOT)

train_size = int(0.8 * len(dataset))
val_size = len(dataset) - train_size
train_dataset, val_dataset = random_split(
    dataset,
    [train_size, val_size],
    generator=torch.Generator().manual_seed(42)
)

train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True, num_workers=2, pin_memory=True)
val_loader = DataLoader(val_dataset, batch_size=16, shuffle=False, num_workers=2, pin_memory=True)

print(len(train_dataset), len(val_dataset))
full, face, hand, labels = next(iter(train_loader))
print(full.shape, face.shape, hand.shape, labels.shape)


17939 4485


/tmp/ipython-input-4179647235.py:21: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  data = torch.load(path)
/tmp/ipython-input-4179647235.py:21: FutureWarning: You are using 

torch.Size([16, 3, 224, 224]) torch.Size([16, 3, 224, 224]) torch.Size([16, 3, 224, 224]) torch.Size([16])


In [8]:
import torchvision.models as models
import torch.optim as optim
import torch.nn as nn


mobilenet = models.mobilenet_v3_small(weights="IMAGENET1K_V1")
mobilenet.classifier = nn.Identity()

model = DriverActionClassifier(backbone=mobilenet, num_classes=10).to(device)

criterion = nn.CrossEntropyLoss()
optimizer = optim.AdamW(model.parameters(), lr=1e-4)
best_val_loss = float('inf')


Downloading: "https://download.pytorch.org/models/mobilenet_v3_small-047dcff4.pth" to /root/.cache/torch/hub/checkpoints/mobilenet_v3_small-047dcff4.pth
100%|██████████| 9.83M/9.83M [00:00<00:00, 198MB/s]


In [9]:
SAVE_PATH = "/content/drive/MyDrive/driver_action_deploy.pth"

for epoch in range(20):
    model.train()
    total_loss = 0
    for full, face, hand, labels in train_loader:
        full, face, hand, labels = full.to(device), face.to(device), hand.to(device), labels.to(device)
        logits = model(full, face, hand)
        loss = criterion(logits, labels)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    avg_train_loss = total_loss / len(train_loader)

    model.eval()
    val_loss = 0
    correct = 0
    total = 0
    with torch.no_grad():
        for full, face, hand, labels in val_loader:
            full, face, hand, labels = full.to(device), face.to(device), hand.to(device), labels.to(device)
            logits = model(full, face, hand)
            loss = criterion(logits, labels)
            val_loss += loss.item()
            pred = logits.argmax(dim=1)
            correct += (pred == labels).sum().item()
            total += labels.size(0)
    avg_val_loss = val_loss / len(val_loader)
    val_acc = correct / total
    if avg_val_loss < best_val_loss:
        best_val_loss = avg_val_loss
        torch.save(model.state_dict(), SAVE_PATH)

    print(f"Epoch {epoch+1}: Train Loss={avg_train_loss:.4f} Val Loss={avg_val_loss:.4f} Val Acc={val_acc:.4f}")


/tmp/ipython-input-4179647235.py:21: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  data = torch.load(path)
/tmp/ipython-input-4179647235.py:21: FutureWarning: You are using 

Epoch 1: Train Loss=0.6147 Val Loss=0.1810 Val Acc=0.9476


/tmp/ipython-input-4179647235.py:21: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  data = torch.load(path)
/tmp/ipython-input-4179647235.py:21: FutureWarning: You are using 

Epoch 2: Train Loss=0.0748 Val Loss=0.0866 Val Acc=0.9775


/tmp/ipython-input-4179647235.py:21: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  data = torch.load(path)
/tmp/ipython-input-4179647235.py:21: FutureWarning: You are using 

...

Epoch 18: Train Loss=0.0031 Val Loss=0.0219 Val Acc=0.9946


/tmp/ipython-input-4179647235.py:21: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  data = torch.load(path)
/tmp/ipython-input-4179647235.py:21: FutureWarning: You are using 

Epoch 19: Train Loss=0.0028 Val Loss=0.0235 Val Acc=0.9938


/tmp/ipython-input-4179647235.py:21: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  data = torch.load(path)
/tmp/ipython-input-4179647235.py:21: FutureWarning: You are using 

Epoch 20: Train Loss=0.0057 Val Loss=0.0252 Val Acc=0.9935


In [11]:
import numpy as np
from sklearn.metrics import confusion_matrix, classification_report

all_preds = []
all_labels = []

with torch.no_grad():
    for full, face, hand, labels in val_loader:
        full, face, hand = full.to(device), face.to(device), hand.to(device)
        logits = model(full, face, hand)
        preds = logits.argmax(dim=1).cpu().numpy()
        all_preds.extend(preds)
        all_labels.extend(labels.numpy())

print(classification_report(all_labels, all_preds))
print(confusion_matrix(all_labels, all_preds))


/tmp/ipython-input-4179647235.py:21: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  data = torch.load(path)
/tmp/ipython-input-4179647235.py:21: FutureWarning: You are using 

              precision    recall  f1-score   support

           0       0.19      0.07      0.10       479
           1       0.00      0.00      0.00       469
           2       0.16      0.06      0.09       462
           3       0.13      0.53      0.21       473
           4       0.11      0.26      0.15       477
           5       0.14      0.27      0.18       448
           6       0.07      0.02      0.03       484
           7       0.00      0.00      0.00       392
           8       0.00      0.00      0.00       372
           9       0.00      0.00      0.00       429

    accuracy                           0.13      4485
   macro avg       0.08      0.12      0.08      4485
weighted avg       0.08      0.13      0.08      4485

[[ 33   0  24 170 139  74  28   2   7   2]
 [ 16   0  14 234  97  88  12   2   5   1]
 [ 25   0  30 199  99  75  28   1   5   0]
 [ 29   1  14 250 102  56  12   6   3   0]
 [ 22   0  26 202 122  87  10   2   5   1]
 [  7   0  19 174 114 119 

In [12]:
idx = np.random.randint(len(val_dataset))
full, face, hand, label = val_dataset[idx]

with torch.no_grad():
    logits = model(
        full.unsqueeze(0).to(device),
        face.unsqueeze(0).to(device),
        hand.unsqueeze(0).to(device)
    )
    pred = logits.argmax(dim=1).item()

print("GT:", label, "Pred:", pred)


/tmp/ipython-input-4179647235.py:21: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  data = torch.load(path)


GT: 8 Pred: 3


In [13]:
def predict_driver_action(model, full, face, hand):
    model.eval()
    with torch.no_grad():
        logits = model(
            full.unsqueeze(0),
            face.unsqueeze(0),
            hand.unsqueeze(0)
        )
        return logits.softmax(dim=1).cpu().numpy()[0]


In [14]:
def preprocess_frame(img, img_size=224):
    img = cv2.resize(img, (img_size, img_size))
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    img = img.astype(np.float32) / 255.0
    img = torch.from_numpy(img).permute(2,0,1)
    mean = torch.tensor([0.485, 0.456, 0.406]).view(3,1,1)
    std = torch.tensor([0.229, 0.224, 0.225]).view(3,1,1)
    return ((img - mean) / std).unsqueeze(0)
